# WhisperWatch second recognizer feasibility pilot

Use a **hosted Colab GPU**. This pilot uses 80 fixed English development clips from separate tuning and verification speakers. It deliberately balances high and low error, the original hybrid's flagged and unflagged clips, and shorter and longer durations. Use the results to judge feasibility and inspect examples. The sample cannot estimate population precision or recall at a 20% budget. The original logistic baseline previously scored the official test. This notebook uses development clips.

Have the pilot code ZIP, `whisperwatch-boosted-results.zip`, and `whisperwatch-ensemble-speaker-split-results.zip` ready. Run each step in order.

## 1. Install tools and freeze the manifest

In [ ]:
%pip install -q "pandas==2.2.3" "pyarrow>=23.0.1" "scikit-learn==1.8.0" "joblib==1.5.3" "av==16.0.1"

In [ ]:
import json
import sys
import pandas as pd
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

from google.colab import files
from IPython.display import display

PROJECT = Path("/content/whisperwatch_second_asr")
SOURCE_FILES = ("__init__", "asr", "features", "inference", "train",
                "ensemble_experiment", "second_asr_pilot")

def upload_one(prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select one ZIP.")
    return BytesIO(next(iter(uploaded.values())))

with ZipFile(upload_one("Upload whisperwatch-second-asr-pilot-code.zip.")) as archive:
    for module in SOURCE_FILES:
        filename = f"src/{module}.py"
        path = PROJECT / filename
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(archive.read(filename))

sys.path.insert(0, str(PROJECT))
from src.second_asr_pilot import REVISION, select_clips

boosted = upload_one("Upload whisperwatch-boosted-results.zip.")
split = upload_one("Upload whisperwatch-ensemble-speaker-split-results.zip.")
selected = select_clips(boosted, split)
manifest_columns = ["clip_id", "speaker_id", "partition", "duration_s", "duration_band",
                    "hybrid_flagged", "true_high_error", "audio_ref", "live_whisper_sample"]
manifest_path = PROJECT / "selected_clips.csv"
selected[manifest_columns].to_csv(manifest_path, index=False)
print(f"Pinned Whispered revision {REVISION}")
print(f"Recorded {len(selected)} clips from {selected.speaker_id.nunique()} different speakers.")
print(f"Selected audio duration {selected.duration_s.sum():.1f} seconds.")
display(selected.groupby(["partition", "hybrid_flagged", "true_high_error", "duration_band"]).size())

## 2. Recover the selected audio

The pinned source has one English development tar of about **717 MB**. The publisher's helper scans forward through it. This notebook uses the same archive layout, caps the payload at **750 MB**, stops after it finds the last requested member, and keeps only selected MP3s. Picked clips are matched by archive revision, filename and clip ID. Decoded duration and available sample rate are checked against metadata. SHA-256 is compared when the source supplies one. [Whispered audio retrieval](https://huggingface.co/datasets/burakaydinofficial/Whispered#how-to-load)

Save the retrieval report even if the pilot stops. The inference cell requires at least 90% retrieval overall, at least four of five clips in each sampling group, and at least six of the eight preselected live comparison clips.

In [ ]:
from src.second_asr_pilot import retrieve_audio

retrieval, retrieval_summary = retrieve_audio(selected, PROJECT / "audio")
retrieval.to_csv(PROJECT / "retrieval.csv", index=False)
(PROJECT / "retrieval_summary.json").write_text(json.dumps(retrieval_summary, indent=2))
print(json.dumps(retrieval_summary, indent=2))
scored = selected.merge(retrieval, on="clip_id", validate="one_to_one")
retrieved = scored.retrieval_status.eq("ok")
per_group = scored.assign(retrieved=retrieved).groupby(
    ["partition", "true_high_error", "hybrid_flagged", "duration_band"]
).retrieved.sum()
retrieval_ok = (retrieved.mean() >= .90 and per_group.min() >= 4
                and (retrieved & scored.live_whisper_sample).sum() >= 6
                and not retrieval_summary["retrieval_error"])
print(f"Verified {retrieved.sum()}/{len(retrieved)} audio clips. Ready for inference? {retrieval_ok}")

## 3. Run Wav2Vec2 on the hosted GPU

The English checkpoint uses around **378 MB** of model weights. The decoder resamples to mono 16 kHz and gives only audio to the recognizer. Text comparison folds case, strips punctuation and symbols, and collapses whitespace. Apostrophes within words are removed. Disagreement is word edits divided by the longer generated transcript's token count, with zero for two empty transcripts. Reference-based WER is computed separately for diagnostics. [Wav2Vec2 model card](https://huggingface.co/facebook/wav2vec2-base-960h)

The next phase requires at least 90% of the selected clips to produce nonempty normalized output, median second-ASR WER of at most 0.50, median second-pass latency at most one second per clip, and at least six usable clips for the live check. These are provisional feasibility limits for this pilot. Report the measured values alongside the decision.

In [ ]:
if not retrieval_ok:
    raise RuntimeError("Retrieval did not pass. Download results in Step 6 and stop here.")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Change to a hosted GPU runtime and rerun the earlier cells.")

%pip install -q "transformers==5.22.2" "huggingface-hub==1.32.0" "faster-whisper==1.2.1" "ctranslate2==4.9.0"

In [ ]:
if not retrieval_ok:
    raise RuntimeError("Download the retrieval results before proceeding.")
from src.second_asr_pilot import run_wav2vec, tokens

inference, run_cost = run_wav2vec(scored)
scored = scored.merge(inference, on="clip_id", validate="one_to_one")
scored.drop(columns=["audio_path", "audio_ref"]).to_csv(PROJECT / "second_asr_results.csv", index=False)
usable = scored.inference_status.eq("ok") & scored.get(
    "second_asr_text", pd.Series("", index=scored.index)
).fillna("").map(lambda text: bool(tokens(text)))
passed = scored.loc[usable]
median_wer = passed.second_asr_wer.median() if len(passed) else None
median_seconds = passed.second_asr_seconds.median() if len(passed) else None
run_cost.update(
    status_counts=scored.inference_status.value_counts().to_dict(),
    usable_clips=int(usable.sum()), selected_clips=len(scored),
    median_wer=median_wer, median_seconds=median_seconds,
    p95_seconds=float(passed.second_asr_seconds.quantile(.95)) if len(passed) else None,
    total_second_asr_seconds=float(scored.second_asr_seconds.sum()) if len(passed) else None,
)
(PROJECT / "second_asr_cost.json").write_text(json.dumps(run_cost, indent=2))
print(json.dumps(run_cost, indent=2))
second_asr_ok = (usable.mean() >= .90 and median_wer <= .50 and median_seconds <= 1
                 and (usable & scored.live_whisper_sample).sum() >= 6)
print(f"Ready for live compatibility check? {second_asr_ok}")

## 4. Compare archived and live Whisper on up to eight clips

The checkpoint download is about **3.09 GB** once. The current WhisperWatch decoder runs in a separate GPU process with CUDA libraries configured before startup. It uses Large V3, forced English, float16, beam size 5, and VAD off. The archive records model family, runner and precision. Its complete VAD, preprocessing and decoding settings are unavailable, so compare observable outputs without attributing differences to a particular setting.

Check normalized text, segment count, duration, mean log probability and the change in disagreement with Wav2Vec2. Fewer than six successful comparisons, median archive-to-live distance above 0.10, or median absolute change in disagreement above 0.10 are reasons to stop before expansion. Check the per-clip table for individual mismatches.

In [ ]:
if not second_asr_ok:
    raise RuntimeError("Wav2Vec2 did not pass. Download results in Step 6 and stop here.")

%pip install -q "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*"

In [ ]:
if not second_asr_ok:
    raise RuntimeError("Download the second-ASR results before proceeding.")

import os
import shutil
import subprocess
from importlib.util import find_spec

if shutil.disk_usage(PROJECT).free < 4_500_000_000:
    raise RuntimeError("The live check needs at least 4.5 GB of free disk.")
sample = scored.loc[scored.live_whisper_sample & usable, [
    "clip_id", "audio_path", "whisper_text", "second_asr_text"
]]
sample_path = PROJECT / "live_input.json"
sample.to_json(sample_path, orient="records")
live_path = PROJECT / "live_whisper.json"
gpu_env = os.environ.copy()
gpu_env.update(WHISPERWATCH_ASR_DEVICE="cuda", WHISPERWATCH_ASR_COMPUTE_TYPE="float16")
lib_dirs = [str(Path(next(iter(find_spec(name).submodule_search_locations))) / "lib")
            for name in ("nvidia.cublas", "nvidia.cudnn")]
gpu_env["LD_LIBRARY_PATH"] = ":".join(filter(None, [*lib_dirs, gpu_env.get("LD_LIBRARY_PATH")]))
subprocess.run([
    sys.executable, "-c",
    "import sys; from pathlib import Path; from src.second_asr_pilot import run_live_check; "
    "run_live_check(Path(sys.argv[1]), Path(sys.argv[2]))",
    str(sample_path), str(live_path),
], cwd=PROJECT, env=gpu_env, check=True)
live = json.loads(live_path.read_text())
comparison = scored.loc[scored.live_whisper_sample, [
    "clip_id", "whisper_text", "archive_segment_count", "duration_s", "avg_logprob",
    "second_asr_text", "disagreement",
]].merge(pd.DataFrame(live["rows"]), on="clip_id", how="left", validate="one_to_one")
comparison["live_status"] = comparison.live_status.fillna("skipped")
for col in ("live_disagreement", "archive_live_disagreement"):
    if col not in comparison:
        comparison[col] = float("nan")
comparison["absolute_disagreement_change"] = (comparison.live_disagreement - comparison.disagreement).abs()
comparison.to_csv(PROJECT / "live_compatibility.csv", index=False)
display(comparison)
exact = comparison.loc[comparison.live_status.eq("ok")]
compatibility_ok = (len(exact) >= 6 and exact.archive_live_disagreement.median() <= .10
                    and exact.absolute_disagreement_change.median() <= .10)
print(f"Successful live comparisons {len(exact)}/8. Compatibility gate passed? {compatibility_ok}")

## 5. Inspect whether disagreement adds information

Compare high-error clips the hybrid missed with low-error clips at similar existing risk and confidence ranks. The matched counts below are descriptive. Pairs can share clips and do not support an independent-sample significance claim. Inspect failures, high disagreement on low-error clips, and similar transcripts that both have high diagnostic WER. The labels and human references enter analysis only.

Expansion requires review of the pilot findings and approval. A later fusion run should sample across both development partitions without using labels, try weights 0%, 10%, 20%, and 30% on tuning speakers, and freeze normalization and the chosen weight before verification. Compare the hybrid, native confidence and fusion on the same top 20%, with a hybrid fallback for missing second-ASR outputs. Report complete-case and all-clip results, false negatives, incremental detections and paired speaker bootstrap intervals. This pilot does not launch that run.

In [ ]:
from src.second_asr_pilot import diagnostics

if "inference_status" not in scored or not scored.inference_status.eq("ok").any():
    raise RuntimeError("No successful second-ASR outputs. Download the available reports in Step 6.")
cohort, matched, examples = diagnostics(scored)
cohort.to_csv(PROJECT / "diagnostic_cohorts.csv")
matched.to_csv(PROJECT / "risk_matched_comparisons.csv", index=False)
examples.drop(columns=["audio_path", "audio_ref"]).to_csv(PROJECT / "diagnostic_examples.csv", index=False)
display(cohort)
display(matched)
display(examples[["example_type", "whisper_text", "second_asr_text", "human_text",
                  "wer_whisper_vs_human", "second_asr_wer", "disagreement"]])
if not second_asr_ok or not globals().get("compatibility_ok", False):
    print("Stop before expansion. Review the failed or incomplete feasibility checks.")
elif matched.empty or not matched.hybrid_flagged.eq(False).any():
    print("The pilot lacks matched evidence about missed clips. Review before proposing expansion.")
else:
    missed = matched.loc[~matched.hybrid_flagged]
    win_share = (missed.high_exceeds_low.sum() + .5 * missed.tied.sum()) / missed.pairs.sum()
    print(f"Descriptive matched win share on clips outside review {win_share:.1%}")
    if win_share > .50:
        print("Investigate further. Review the examples and measured costs before requesting expansion.")
    else:
        print("The pilot did not show a positive matched signal for clips outside review.")

## 6. Download the pilot evidence

In [ ]:
from zipfile import ZIP_DEFLATED
output = PROJECT.parent / "whisperwatch-second-asr-pilot-results.zip"
with ZipFile(output, "w", compression=ZIP_DEFLATED) as report:
    for artifact in sorted(PROJECT.iterdir()):
        if artifact.is_file() and artifact.suffix in {".csv", ".json"} and artifact.name != "live_input.json":
            report.write(artifact, artifact.name)
files.download(str(output))